# Laszlo Cseh vs Michael Phelps

In this notebook we will be analyzing the careers of Michael Phelps and Laszlo Cseh.

#### Step 1 Loading Datasets
We will start off by getting the data from the web, and altering the columns to match what we want.

In [74]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

url = "https://api.worldaquatics.com/fina/athletes/1001621/results"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
data = response.json()

# Convert to pandas DataFrame
michael_Phelps = pd.DataFrame(data["Results"])

michael_Phelps.head(3)

,Rank,MedalTag,SportCode,DisciplineName,PhaseName,RecordType,NAT,CompetitionName,CompetitionType,CompetitionCountry,CompetitionCity,Date,Time,UtcDateTime,Points,Tags,AthleteResultAge
0,1.0,G,SW,Men 4x100m Medley Relay,Finals,OR,USA,Olympic Games Rio 2016,Olympic Games,BRA,Rio de Janeiro,2016-08-13,3:27.95,2016-08-14T02:04:00,990.0,None,31
1,2.0,S,SW,Men 100m Butterfly,Finals,,USA,Olympic Games Rio 2016,Olympic Games,BRA,Rio de Janeiro,2016-08-12,51.14,2016-08-13T01:12:00,924.0,None,31
2,1.0,G,SW,Men 200m Medley,Finals,,USA,Olympic Games Rio 2016,Olympic Games,BRA,Rio de Janeiro,2016-08-11,1:54.66,2016-08-12T02:01:00,982.0,None,31


In [75]:
url = "https://api.worldaquatics.com/fina/athletes/1007320/results"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
data = response.json()

# Convert to pandas DataFrame
laszlo_Cseh = pd.DataFrame(data["Results"])

laszlo_Cseh.head(3)

,Rank,MedalTag,SportCode,DisciplineName,PhaseName,RecordType,NAT,CompetitionName,CompetitionType,CompetitionCountry,CompetitionCity,Date,Time,UtcDateTime,Points,Tags,AthleteResultAge,ClubName
0,7.0,None,SW,Men 200m Medley,Finals,,HUN,Olympic Games Tokyo 2020,Olympic Games,JPN,Tokyo,2021-07-30,1:57.68,2021-07-30T02:20:15,909.0,None,35,NaN
1,2.0,None,SW,Men 200m Freestyle,Finals,,HUN,Budapest Open Championships,Open,HUN,Budapest,2021-06-19,1:49.34,NaN,811.0,None,35,Budapesti Vasutas Sport Club
2,2.0,None,SW,Men 100m Butterfly,Finals,,HUN,Budapest Open Championships,Open,HUN,Budapest,2021-06-19,52.57,NaN,834.0,None,35,Budapesti Vasutas Sport Club


In [76]:
michael_Phelps = michael_Phelps.rename(columns={
    "Rank": "Overall Rank",
    "DisciplineName": "Event",
    "Time": "Time",
    "Points": "Points",
    "Tags": "Tag",
    "AthleteResultAge": "Age",
    "CompetitionName": "Competition",
    "CompetitionCountry": "Comp Country",
    "Date": "Date"
})

michael_Phelps = michael_Phelps[
    ["Overall Rank", "Event", "Time", "RecordType", "Age", "Competition", "Comp Country", "Date"]
]
michael_Phelps.head(3)

,Overall Rank,Event,Time,RecordType,Age,Competition,Comp Country,Date
0,1.0,Men 4x100m Medley Relay,3:27.95,OR,31,Olympic Games Rio 2016,BRA,2016-08-13
1,2.0,Men 100m Butterfly,51.14,,31,Olympic Games Rio 2016,BRA,2016-08-12
2,1.0,Men 200m Medley,1:54.66,,31,Olympic Games Rio 2016,BRA,2016-08-11


In [78]:
laszlo_Cseh = laszlo_Cseh.rename(columns={
    "Rank": "Overall Rank",
    "DisciplineName": "Event",
    "Time": "Time",
    "Points": "Points",
    "Tags": "Tag",
    "AthleteResultAge": "Age",
    "CompetitionName": "Competition",
    "CompetitionCountry": "Comp Country",
    "Date": "Date"
})

laszlo_Cseh = laszlo_Cseh[
    ["Overall Rank", "Event", "Time", "RecordType", "Age", "Competition", "Comp Country", "Date"]
]
laszlo_Cseh.head(3)

,Overall Rank,Event,Time,RecordType,Age,Competition,Comp Country,Date
0,7.0,Men 200m Medley,1:57.68,,35,Olympic Games Tokyo 2020,JPN,2021-07-30
1,2.0,Men 200m Freestyle,1:49.34,,35,Budapest Open Championships,HUN,2021-06-19
2,2.0,Men 100m Butterfly,52.57,,35,Budapest Open Championships,HUN,2021-06-19


In [68]:
michael_Phelps["swimmer"] = "Michael Phelps"
laszlo_Cseh["swimmer"] = "Laszlo Cseh"

combined_df = pd.concat(
    [michael_Phelps, laszlo_Cseh],
    ignore_index=True
)
event_counts = (
    combined_df
        .groupby(["Event", "swimmer"])
        .size()
        .unstack(fill_value=0)
)
event_counts["total"] = event_counts.sum(axis=1)

event_counts = event_counts.sort_values("total", ascending=False)

event_counts.head(10)

swimmer,Laszlo Cseh,Michael Phelps,total
Event,,,
Men 200m Medley,82,45,127
Men 200m Butterfly,82,42,124
Men 100m Butterfly,68,45,113
Men 400m Medley,58,24,82
Men 50m Butterfly,52,4,56
Men 200m Freestyle,26,28,54
Men 4x100m Medley Relay,15,17,32
Men 100m Freestyle,4,23,27
Men 100m Backstroke,14,9,23


In [71]:
head_to_head = (
    combined_df
        .groupby("Competition")
        .filter(lambda x: {"Michael Phelps", "Laszlo Cseh"}.issubset(x["swimmer"].unique()))
)
vs_counts = (
    head_to_head
        .groupby("Event")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
)
vs_counts

,Event,count
7,Men 200m Medley,18
5,Men 200m Butterfly,15
9,Men 400m Medley,14
2,Men 100m Butterfly,13
12,Men 4x200m Freestyle Relay,11
11,Men 4x100m Medley Relay,10
10,Men 4x100m Freestyle Relay,8
6,Men 200m Freestyle,7
1,Men 100m Backstroke,3
4,Men 200m Backstroke,1


In [ ]:
michael_Phelps["Date"] = pd.to_datetime(michael_Phelps["Date"])


def time_to_seconds(t):
    if pd.isna(t):
        return np.nan
    if isinstance(t, str):
        t = t.strip()
        if t in ["DNS", "DNF", "DSQ"]:
            return np.nan
        if ":" in t:
            m, s = t.split(":")
            return int(m) * 60 + float(s)
    return float(t)

michael_Phelps["Time_seconds"] = michael_Phelps["Time"].apply(time_to_seconds)

event_name = "Men 200m Medley"
event_michael_Phelps= michael_Phelps[michael_Phelps["Event"] == event_name].sort_values("Date")

laszlo_Cseh["Date"] = pd.to_datetime(laszlo_Cseh["Date"])


def time_to_seconds(t):
    if pd.isna(t):
        return np.nan
    if isinstance(t, str):
        t = t.strip()
        if t in ["DNS", "DNF", "DSQ"]:
            return np.nan
        if ":" in t:
            m, s = t.split(":")
            return int(m) * 60 + float(s)
    return float(t)

laszlo_Cseh["Time_seconds"] = laszlo_Cseh["Time"].apply(time_to_seconds)

event_name = "Men 200m Medley"
event_laszlo_Cseh= laszlo_Cseh[laszlo_Cseh["Event"] == event_name].sort_values("Date")


In [80]:
events = [
    "Men 400m Medley",
    "Men 200m Butterfly",
    "Men 100m Butterfly",
    "Men 200m Medley",
    "Men 100m Backstroke"
]
def plot_event_comparison(event_name, df_phelps, df_cseh, window=5):
    # Filter event
    p = df_phelps[df_phelps["Event"] == event_name].sort_values("Date").copy()
    c = df_cseh[df_cseh["Event"] == event_name].sort_values("Date").copy()

    if p.empty and c.empty:
        print(f"No data for {event_name}")
        return

    # Rolling means
    p["Rolling_mean"] = p["Time_seconds"].rolling(window, center=True).mean()
    c["Rolling_mean"] = c["Time_seconds"].rolling(window, center=True).mean()

    plt.figure(figsize=(10, 6))
    sns.set(style="whitegrid")

    # Michael Phelps
    if not p.empty:
        sns.lineplot(
            data=p, x="Date", y="Time_seconds",
            color="tab:blue", alpha=0.3, marker="o",
            label="Phelps (races)"
        )
        sns.lineplot(
            data=p, x="Date", y="Rolling_mean",
            color="tab:blue", linewidth=3,
            label="Phelps (rolling mean)"
        )

    # László Cseh
    if not c.empty:
        sns.lineplot(
            data=c, x="Date", y="Time_seconds",
            color="tab:orange", alpha=0.3, marker="o",
            label="Cseh (races)"
        )
        sns.lineplot(
            data=c, x="Date", y="Rolling_mean",
            color="tab:orange", linewidth=3,
            label="Cseh (rolling mean)"
        )

    plt.gca().invert_yaxis()
    plt.title(f"{event_name} – Performance Over Time")
    plt.xlabel("Date")
    plt.ylabel("Time (seconds)")
    plt.legend()
    plt.tight_layout()
    plt.show()
for event in events:
    plot_event_comparison(event, michael_Phelps, laszlo_Cseh, window=5)


KeyError: 'Time_seconds'

In [79]:
events.head()

AttributeError: 'list' object has no attribute 'head'